In [4]:
# This code is used to calculate the average contact between DNA End1 with Histone Tail1 and DNA End2 with Histone Tail2 at each window. 
# Finally it produced a plot where x axis will be Rg value of each window whil y axis will be average contact. The final plot consist of 
# average contact for each tail-dna and as well as total contact (sum of both contacts contributing from tail1-dna end1 and tail2-dna end2).
# Software used: MD Analysis

In [ ]:
# This chunk of code produce separate file of contact values for each window.

import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
from MDAnalysis.analysis import distances

top = "nucl.pdb"
cutoff = 4.5
nblocks = 8
umbrellas = range(13)  # umb0 to umb12

for i in umbrellas:
    traj = f"umb{i}_upto50ns.nucl.pbc.xtc"
    u = mda.Universe(top, traj)

    # selections (heavy atoms only)
    h3_tail1 = u.select_atoms("segid C and resid 1:44 and not name H*")
    h3_tail2 = u.select_atoms("segid G and resid 1:44 and not name H*")
    dna_end1 = u.select_atoms("(chainID A and resid 118:147 or chainID B and resid 1:30) and not name H*")
    dna_end2 = u.select_atoms("(chainID A and resid 1:30 or chainID B and resid 118:147) and not name H*")

    results = []
    for ts in u.trajectory:
        dist1 = distances.distance_array(h3_tail1.positions, dna_end1.positions)
        dist2 = distances.distance_array(h3_tail2.positions, dna_end2.positions)
        contacts1 = np.count_nonzero(dist1 <= cutoff)
        contacts2 = np.count_nonzero(dist2 <= cutoff)
        total = contacts1 + contacts2
        results.append([ts.frame + 1, contacts1, contacts2, total])

    np.savetxt(f"h3_dna_contacts_umb{i}.dat", results, fmt="%d",
               header="Frame Contacts1 Contacts2 Total", comments="")

    # plotting
    data = np.array(results)
    frames = data[:, 0]
    contacts1 = data[:, 1]
    contacts2 = data[:, 2]
    total = data[:, 3]

    plt.figure(figsize=(8, 5))
    plt.plot(frames, contacts1, label="H3_tail1–DNA_end1")
    plt.plot(frames, contacts2, label="H3_tail2–DNA_end2")
    plt.plot(frames, total, label="Total Contacts", linewidth=2)
    plt.xlabel("Frame")
    plt.ylabel("Number of Contacts")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"h3_dna_contacts_umb{i}.png", dpi=300)
    plt.close()

    # block average
    def block_average(values, nblocks=nblocks):
        n = len(values)
        block_size = n // nblocks
        blocks = [values[i*block_size:(i+1)*block_size].mean() for i in range(nblocks)]
        mean = np.mean(blocks)
        stderr = np.std(blocks, ddof=1) / np.sqrt(nblocks)
        return mean, stderr

    avg1, err1 = block_average(contacts1)
    avg2, err2 = block_average(contacts2)
    avgt, errt = block_average(total)

    with open(f"h3_dna_contacts_avg_umb{i}.dat", "w") as f:
        f.write("#ContactType\tAverage\tError\n")
        f.write(f"H3_tail1–DNA_end1\t{avg1:.3f}\t{err1:.3f}\n")
        f.write(f"H3_tail2–DNA_end2\t{avg2:.3f}\t{err2:.3f}\n")
        f.write(f"Total\t{avgt:.3f}\t{errt:.3f}\n")

    print(f"Completed umbrella {i}")


In [ ]:
# This will provide summary data for all windows in single file.

import numpy as np

umbrellas = range(13)  # umb0 to umb12
summary = []

for i in umbrellas:
    avg_file = f"h3_dna_contacts_avg_umb{i}.dat"
    with open(avg_file, "r") as f:
        lines = f.readlines()[1:]  # skip header
        tail1_line = [line for line in lines if line.startswith("H3_tail1")][0]
        tail2_line = [line for line in lines if line.startswith("H3_tail2")][0]
        total_line = [line for line in lines if line.startswith("Total")][0]

        _, avg1, err1 = tail1_line.strip().split("\t")
        _, avg2, err2 = tail2_line.strip().split("\t")
        _, avgt, errt = total_line.strip().split("\t")

        summary.append([i, float(avg1), float(err1),
                           float(avg2), float(err2),
                           float(avgt), float(errt)])

summary = np.array(summary)
np.savetxt("h3_dna_contacts_summary_all.dat", summary, 
           fmt="%d %.3f %.3f %.3f %.3f %.3f %.3f",
           header="Umbrella Tail1_Avg Tail1_Error Tail2_Avg Tail2_Error Total_Avg Total_Error",
           comments="")



In [ ]:
# This will provide the final figure.


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# load summary
data = np.loadtxt("h3_dna_contacts_summary_all.dat", skiprows=1)
columns = ["Umbrella", "Tail1_Avg", "Tail1_Error", "Tail2_Avg", "Tail2_Error", "Total_Avg", "Total_Error"]
df = pd.DataFrame(data, columns=columns)

# ---- Plot setup ----
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Plot Tail1
ax.errorbar(df["Umbrella"], df["Tail1_Avg"], yerr=df["Tail1_Error"], fmt='o-', label="Tail1–DNA_end1", 
            markersize=8, linewidth=2, capsize=4)

# Plot Tail2
ax.errorbar(df["Umbrella"], df["Tail2_Avg"], yerr=df["Tail2_Error"], fmt='s-', label="Tail2–DNA_end2", 
            markersize=8, linewidth=2, capsize=4)

# Plot Total
ax.errorbar(df["Umbrella"], df["Total_Avg"], yerr=df["Total_Error"], fmt='^-', label="Total Contacts", 
            markersize=8, linewidth=2, capsize=4)

# ---- Styling ----
ax.set_xlabel("Rg", color='black', fontsize=22, fontweight='bold')
ax.set_ylabel("Nc", color='black', fontsize=22, fontweight='bold')
ax.tick_params(axis='x', labelsize=18, width=2, length=6, colors='black', direction='in')
ax.tick_params(axis='y', labelsize=18, width=2, length=6, colors='black', direction='in')

# Border styling
for spine in ax.spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

ax.legend(prop={'weight': 'bold', 'size': 12}, loc='upper right')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("h3_dna_contacts_summary_plot_styled.png", dpi=600)
plt.show()
